For a given subject, we want:
    1. Compute the run-pairwise correlation of tmaps (how similar are activity patterns between runs)
    2. Compute the run-pairwise correlation of FD (how similar are motion pattern between runs)
    3. Correlate these two matrices (how similar is activity similarity to head motion similarity)
        If one method results in a lower similarity, it is better
        


In [110]:
from mreyemove.analysis.glm.config import GLMConfig
import numpy as np
from SleepMReye.sleepmreye.sleep_mrio import SleepMRIO

"""
Only keep subjects who have at least two runs with 50% awake states
"""

signal = "clean"

config = GLMConfig.from_yaml("/Users/zach/2025_RA/matthias/eyemove_and_sleep/SleepMReye/designs/original.yml")
dataset = SleepMRIO(
    bids_root="/Users/zach/2025_RA/matthias/bids_sleepmreye",
    exclude_subjects=["01", "07", "09", "12"],
    tr=2.1,
    task="sleep",
    load_signal=True,
    desc_add=signal,
)

def proportion_awake(run_df, state: str):
    return (run_df["state"] == state).sum() / len(run_df)

subjects_to_keep = []
threshold = 1/427
state = "W"
contrast = "mreyemove"


for subject in dataset.subjects:
    n_awake = 0
    
    for subject, run, run_df in dataset.iter_run_events(subject):
        if proportion_awake(run_df, state) > threshold:
            n_awake += 1
         # however you get this
        # print(f"Subject {subject}, Run {run}: {n_awake_trs}/427 awake TRs ({n_awake_trs/427*100:.1f}%)")
    if n_awake < 3:
      drop = "Dropping"
    else:
        drop = "Keeping"
        subjects_to_keep.append(subject)
    print(f"{drop} subject {subject}: {n_awake} awake > {threshold} runs")
print(len(subjects_to_keep))
        

Keeping subject 10: 6 awake > 0.00234192037470726 runs
Keeping subject 22: 7 awake > 0.00234192037470726 runs
Keeping subject 06: 4 awake > 0.00234192037470726 runs
Keeping subject 18: 5 awake > 0.00234192037470726 runs
Keeping subject 25: 6 awake > 0.00234192037470726 runs
Keeping subject 17: 7 awake > 0.00234192037470726 runs
Keeping subject 33: 7 awake > 0.00234192037470726 runs
Keeping subject 19: 3 awake > 0.00234192037470726 runs
Keeping subject 24: 3 awake > 0.00234192037470726 runs
Keeping subject 16: 3 awake > 0.00234192037470726 runs
Keeping subject 32: 8 awake > 0.00234192037470726 runs
Keeping subject 08: 7 awake > 0.00234192037470726 runs
Keeping subject 11: 5 awake > 0.00234192037470726 runs
Keeping subject 23: 3 awake > 0.00234192037470726 runs
Keeping subject 26: 6 awake > 0.00234192037470726 runs
Keeping subject 30: 5 awake > 0.00234192037470726 runs
Keeping subject 14: 5 awake > 0.00234192037470726 runs
Keeping subject 29: 7 awake > 0.00234192037470726 runs
Keeping su

In [111]:
methods = ["raw-conv-fd", "clean-original"]

method_rs = {}

df = dataset.events

for method in methods:
    rs = []
    
    for subject in subjects_to_keep:
        tmaps = []
        fds = []
        n_runs_used = 0
        # print(f"Processing subject {subject}")
        runs = df[df["subject"] == subject]["run"].unique()
    
        for run in runs:
            run_df = df[(df["subject"] == subject) & (df["run"] == run)]
            if proportion_awake(run_df, state) < threshold:
                continue
            n_runs_used += 1
            # Load and compute t maps
            run_data, _  = dataset.load_run_level_glm_data(subject=subject, run=run, desc_add=method, contrasts=config.contrasts)
            cope_data = run_data[contrast].cope.get_fdata()
            varcope_data = run_data[contrast].varcope.get_fdata()
            
            # Calculate t-statistic: t = COPE / sqrt(VARCOPE)
            # Add small epsilon to avoid division by zero
            epsilon = 1e-10
            tstat_data = cope_data / np.sqrt(varcope_data + epsilon)
            tmaps.append(tstat_data.ravel())
            
            # Load FDs 
            fd = dataset.load_confounds(subject=subject, run=run, confound_names=["framewise_displacement"]).to_numpy()
            fds.append(fd.ravel())
            
        print(f"Using {n_runs_used} runs for subject {subject}")
        # Stack into (n_runs, n_voxels) array
        tmaps_2d = np.array(tmaps)
        fds_2d = np.array(fds)
        
        # # Compute correlation matrix
        tmap_corr_matrix = np.corrcoef(tmaps_2d)
        fds_corr_matrix = np.corrcoef(fds_2d)
        # 
        # # Extract upper triangle
        triu_idx = np.triu_indices(n_runs_used, k=1)
        tmap_vec = tmap_corr_matrix[triu_idx]
        fd_vec = fds_corr_matrix[triu_idx]
         
        rs.append(np.corrcoef(tmap_vec, fd_vec)[0,1])
        
    method_rs[method] = np.array(rs)

Using 6 runs for subject 10
Using 7 runs for subject 22
Using 4 runs for subject 06
Using 5 runs for subject 18
Using 6 runs for subject 25
Using 7 runs for subject 17
Using 7 runs for subject 33
Using 3 runs for subject 19
Using 3 runs for subject 24
Using 3 runs for subject 16
Using 8 runs for subject 32
Using 7 runs for subject 08
Using 5 runs for subject 11
Using 3 runs for subject 23
Using 6 runs for subject 26
Using 5 runs for subject 30
Using 5 runs for subject 14
Using 7 runs for subject 29
Using 4 runs for subject 13
Using 6 runs for subject 05
Using 7 runs for subject 04
Using 5 runs for subject 20
Using 6 runs for subject 27
Using 3 runs for subject 03
Using 5 runs for subject 31
Using 5 runs for subject 15
Using 6 runs for subject 28
Using 6 runs for subject 10
Using 7 runs for subject 22
Using 4 runs for subject 06
Using 5 runs for subject 18
Using 6 runs for subject 25
Using 7 runs for subject 17
Using 7 runs for subject 33
Using 3 runs for subject 19
Using 3 runs for sub

In [112]:
def fisher_z(r):
    """Fisher Z-transformation"""
    return 0.5 * np.log((1 + r) / (1 - r))



# Transform before testing
z_method1 = fisher_z(np.array(method_rs[methods[0]]))
z_method2 = fisher_z(np.array(method_rs[methods[1]]))

# Now do paired t-test on z-scores
from scipy.stats import ttest_rel, ttest_1samp
t_stat, p_value = ttest_rel(z_method1, z_method2)

print(f"{methods[0]} mean r: {np.mean(method_rs[methods[0]]):.3f} (z={np.mean(z_method1):.3f})")
print(f"{methods[1]} mean r: {np.mean(method_rs[methods[1]]):.3f} (z={np.mean(z_method2):.3f})")
print(f"Paired t-test on Fisher Z: t={t_stat:.3f}, p={p_value:.4f}")

for i in range(2):
    z_method1 = fisher_z(np.array(method_rs[methods[i]]))
    t_stat, p_value = ttest_1samp(z_method1, 0)
    # print(f"1 sample t-test for method {methods[i]} on Fisher Z: t={t_stat:.3f}, p={p_value:.4f}")
    mean_r = np.mean(method_rs[methods[i]])
    print(f"{methods[i]}: mean r = {mean_r:.3f}, t={t_stat:.3f}, p={p_value:.4f}")


raw-conv-fd mean r: 0.048 (z=0.076)
clean-original mean r: 0.006 (z=0.015)
Paired t-test on Fisher Z: t=1.293, p=0.2075
raw-conv-fd: mean r = 0.048, t=0.759, p=0.4547
clean-original: mean r = 0.006, t=0.156, p=0.8772
